# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example walkthrough for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All records, fields, and columns are referenced by their unique `@id` as defined in the Croissant schema.

### Dataset Source
The dataset (Croissant schema) is available at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and initialize the Croissant interface using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and initialize Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Next, discover what record sets, fields, and columns are available in the dataset. All are referenced by their `@id` (the identifier string in the Croissant schema).

In [ ]:
# List record sets by their '@id' and show summary of columns/fields for each
record_sets = dataset.metadata.record_sets
print('Record Sets:')
for rset in record_sets:
    print(f"- @id: {rset.id} | name: {rset.name if hasattr(rset, 'name') else 'N/A'}")
    if hasattr(rset, 'fields'):
        for f in rset.fields:
            print(f"    - Field @id: {f.id} | name: {f.name if hasattr(f, 'name') else 'N/A'} | dataType: {getattr(f, 'data_type', None)}")
    else:
        print('    No fields listed for this record set.')

## 3. Data Extraction
Now, extract records from a selected record set. Replace the shown example `@id` values with those found in the previous step to explore other record sets.

For this notebook, we will focus on the first available record set (if present).

In [ ]:
# If there is at least one record set, extract their data
dataframes = {}
if record_sets:
    selected_record_set = record_sets[0]  # just pick the first one for demonstration
    selected_record_set_id = selected_record_set.id
    print(f"Extracting records for record set '@id': {selected_record_set_id}")
    # Load records
    records = list(dataset.records(record_set=selected_record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[selected_record_set_id] = df
        print('Fields (columns) in DataFrame:')
        print(df.columns.tolist())
        display(df.head())
    else:
        print('No records found for the selected record set.')
else:
    print('No record sets found in the dataset.')

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing – filtering, normalization, and grouping. All operations should reference fields by their `@id`.

**Note:** The specific field `@id` values must match the schema; see Section 2 for possible candidates.

In [ ]:
import numpy as np
# For demonstration, choose the first numeric field from the record set fields if it exists
if record_sets:
    # Try to select a numeric field from field metadata
    numeric_field_id = None
    group_field_id = None
    for f in selected_record_set.fields:
        if hasattr(f, 'data_type') and (f.data_type in ['Integer', 'Float', 'Number']):
            numeric_field_id = f.id
            break
    # Also try to pick a string/categorical field for grouping
    for f in selected_record_set.fields:
        if hasattr(f, 'data_type') and (f.data_type == 'Text' or f.data_type == 'String'):
            group_field_id = f.id
            break
    df = dataframes.get(selected_record_set_id)
    if numeric_field_id and numeric_field_id in df.columns:
        print(f"Selected numeric field for EDA: {numeric_field_id}")
        series = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = np.nanpercentile(series, 75)
        filtered_df = df[series > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        display(filtered_df[[numeric_field_id]].head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (series[series > threshold] - series.mean()) / series.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by the group_field_id if available
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print('No suitable grouping field found.')
    else:
        print('No numeric field detected in record set for EDA.')
else:
    print('No record sets available for EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field or any relationship with the grouping field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    series = pd.to_numeric(df[numeric_field_id], errors='coerce').dropna()
    sns.histplot(series, bins=30, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} values")
    plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook has demonstrated how to load a FAIR dataset described via [Croissant](https://mlcommons.org/croissant/) using the `mlcroissant` library. You explored how to:
- Discover record sets and enumerate their fields/columns by `@id`
- Load records into DataFrames by record set `@id`
- Apply basic filtering, normalization, and group-wise analysis referencing the Croissant schema structure
- Visualize distributions and grouped data

Feel free to extend this notebook to additional record sets or fields by referencing their `@id` as found in the schema overview (Section 2).